# Phase 3 — Data Cleaning & Preprocessing
Stack Overflow Developer Survey 2025.

**Steps:**
1. Drop rows with null salary
2. Remove salary outliers (< 0k or > 00k)
3. Filter to full-time employed respondents only
4. Cap erroneous WorkExp values (> 50 years)
5. Handle missing values — numeric: median, categorical: "Unknown"
6. Remove duplicate rows
7. Save cleaned dataset to 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 60)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## Load Raw Data

In [ ]:
df = pd.read_csv('../data/results.txt', low_memory=False)
print(f'Raw shape: {df.shape}')

## Step 1 — Drop Null Salary

In [ ]:
df = df[df['ConvertedCompYearly'].notna()].copy()
print(f'After dropping null salary: {len(df):,} rows')

## Step 2 — Remove Salary Outliers (< 0k or > 00k)

In [ ]:
before = len(df)
df = df[df['ConvertedCompYearly'].between(10_000, 500_000)].copy()
after  = len(df)
print(f'Dropped {before - after:,} outlier rows (< 0k or > 00k)')
print(f'Remaining: {after:,} rows')
print()
print(df['ConvertedCompYearly'].describe().apply(lambda x: f"{x:,.0f}"))

## Step 3 — Filter to Full-Time Employed Only

In [ ]:
print('Employment value counts before filter:')
print(df['Employment'].value_counts())
print()

df = df[df['Employment'] == 'Employed'].copy()
print(f'After filtering to Employed: {len(df):,} rows')

## Step 4 — Cap Erroneous WorkExp Values (> 50 years)

In [ ]:
bad_workexp = (df['WorkExp'] > 50).sum()
print(f'Rows with WorkExp > 50: {bad_workexp}')

df['WorkExp'] = df['WorkExp'].clip(upper=50)
print('WorkExp capped at 50.')
print(df['WorkExp'].describe())

## Step 5 — Handle Missing Values

In [ ]:
key_cols = [
    'ConvertedCompYearly',
    'WorkExp',
    'YearsCode',
    'Country',
    'DevType',
    'EdLevel',
    'Employment',
    'LanguageHaveWorkedWith',
    'DatabaseHaveWorkedWith',
    'WebframeHaveWorkedWith',
]

print('Missing % for key columns (before imputation):')
missing = (df[key_cols].isna().sum() / len(df) * 100).sort_values(ascending=False)
print(missing.apply(lambda x: f"{x:.1f}%").to_string())

In [ ]:
# Numeric columns — median imputation
for col in ['WorkExp', 'YearsCode']:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f'{col}: filled NaN with median = {median_val}')

# Categorical columns — fill with "Unknown"
for col in ['Country', 'DevType', 'EdLevel', 'LanguageHaveWorkedWith',
            'DatabaseHaveWorkedWith', 'WebframeHaveWorkedWith']:
    n_filled = df[col].isna().sum()
    df[col] = df[col].fillna('Unknown')
    print(f"{col}: filled {n_filled:,} NaN with 'Unknown'")

In [ ]:
print('Missing % after imputation:')
missing_after = (df[key_cols].isna().sum() / len(df) * 100).sort_values(ascending=False)
print(missing_after.apply(lambda x: f"{x:.1f}%").to_string())

## Step 6 — Remove Duplicate Rows

In [ ]:
dupes = df.duplicated().sum()
print(f'Duplicate rows found: {dupes}')
df = df.drop_duplicates()
print(f'Remaining rows after dedup: {len(df):,}')

## Step 7 — Final Shape & Save

In [ ]:
print(f'Final cleaned dataset shape: {df.shape}')
print()
print('Salary stats after cleaning:')
print(df['ConvertedCompYearly'].describe().apply(lambda x: f"{x:,.0f}"))

In [ ]:
# Quick sanity plot — salary distribution after cleaning
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['ConvertedCompYearly'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Salary Distribution — Cleaned (raw)')
axes[0].set_xlabel('USD / year')
axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'k'))

log_sal = np.log1p(df['ConvertedCompYearly'])
axes[1].hist(log_sal, bins=60, color='darkorange', edgecolor='white')
axes[1].set_title('Salary Distribution — Cleaned (log1p)')
axes[1].set_xlabel('log(USD + 1)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../reports/salary_distribution_cleaned.png', bbox_inches='tight')
plt.show()

In [ ]:
df.to_csv('../data/cleaned.csv', index=False)
print('Saved to data/cleaned.csv')
print(f'Rows: {len(df):,}  |  Columns: {df.shape[1]}')

## Phase 3 Summary

| Step | Action | Result |
|------|--------|--------|
| 1 | Drop null salary | ~25k rows removed |
| 2 | Remove outliers < 0k or > 00k | Extreme values removed |
| 3 | Filter to Employed only | Freelancers/students removed |
| 4 | Cap WorkExp > 50 | Erroneous 99/100 values fixed |
| 5 | Impute missing values | Numeric → median, Categorical → "Unknown" |
| 6 | Drop duplicates | Deduplicated |
| 7 | Save |  ready for Phase 4 |

> *Fill in actual row counts after running cells above.*